# Notebook 02: Real-World Dataset Exploration (ISOLET)

## Dataset choice rationale
**Definition:** ISOLET is a spoken-letter recognition dataset where each sample is represented by a high-dimensional acoustic feature vector.

**Why chosen:**
- 613 input columns (fits the 500+ feature requirement).
- Real recording-driven data, not synthetic benchmark generation.
- Multi-class setting (26 letters) adds realistic modeling complexity.

**Key tradeoff:** richer realism than synthetic data, but harder optimization and longer benchmark runtimes.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from src.data_loader import load_isolet_dataset

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')

FIG_DIR = Path('outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)


## Data loading and schema validation

**Code explanation:** `load_isolet_dataset()` first checks local parquet cache under `data/real/`, then downloads from OpenML once if needed.


In [ ]:
X, y, metadata = load_isolet_dataset()

print('Metadata:')
for k, v in metadata.items():
    print(f'  {k}: {v}')

print('\nData shape:', X.shape)
print('Target shape:', y.shape)
print('Class count:', y.nunique())


## Data quality audit

We inspect missing values, low-variance columns, and duplicate rows to understand preprocessing risk before modeling.


In [ ]:
quality_report = pd.DataFrame(
    {
        'metric': [
            'rows',
            'features',
            'missing_values_total',
            'duplicate_rows',
            'constant_features',
            'near_constant_features(var<1e-4)',
        ],
        'value': [
            int(X.shape[0]),
            int(X.shape[1]),
            int(X.isna().sum().sum()),
            int(X.duplicated().sum()),
            int((X.var() == 0).sum()),
            int((X.var() < 1e-4).sum()),
        ],
    }
)
quality_report


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

class_counts = y.value_counts().sort_index()
sns.barplot(x=class_counts.index.astype(str), y=class_counts.values, ax=axes[0], color='teal')
axes[0].set_title('Class Distribution (26 spoken letters)')
axes[0].set_xlabel('Class label')
axes[0].set_ylabel('Count')

variance_sample = X.var().sample(min(200, X.shape[1]), random_state=42)
sns.histplot(variance_sample, bins=35, kde=True, ax=axes[1], color='purple')
axes[1].set_title('Feature Variance Distribution (Sampled Columns)')
axes[1].set_xlabel('Variance')

plt.tight_layout()
plt.savefig(FIG_DIR / 'isolet_quality_overview.png', dpi=300, bbox_inches='tight')
plt.show()


## Correlation structure and redundancy intuition

**Theory:** Highly correlated variables can duplicate information and inflate model variance.


In [ ]:
corr_subset = X.sample(min(1200, len(X)), random_state=42).iloc[:, :120].corr(method='spearman').abs()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_subset, cmap='mako', cbar=False)
plt.title('Spearman Correlation Heatmap (120-feature subset)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'isolet_correlation_subset_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## Baseline model before feature selection

**Interpretation goal:** quantify starting performance so later feature-selection improvements are evidence-based.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

baseline_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
baseline_model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, baseline_model.predict(X_train))
test_acc = accuracy_score(y_test, baseline_model.predict(X_test))

pd.DataFrame(
    {
        'split': ['train', 'test'],
        'accuracy': [train_acc, test_acc],
        'n_features': [X_train.shape[1], X_train.shape[1]],
    }
)


## Summary

- ISOLET provides realistic high-dimensional structure (613 features, 26 classes).
- Data quality is strong (no missingness by default), but redundancy and model complexity remain significant.
- We now build a progressive feature-selection funnel to reduce features while preserving predictive quality.
